In [6]:
# Imports
import pandas as pd
import numpy as np
import re
from scipy import stats
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from transformers import DistilBertTokenizer, DistilBertConfig, DistilBertModel
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import os

In [7]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.__version__)


11.8
True
2.7.1+cu118


In [8]:
data = pd.read_csv('content/Social Media Engagement Dataset.csv') # Change path according to your device
# data = data.sample(n=100) # Get 100 samples - Testing purposes


# **Data Exploration**

In [9]:
data.head()

,post_id,timestamp,day_of_week,platform,user_id,location,language,text_content,hashtags,mentions,...,comments_count,impressions,engagement_rate,brand_name,product_name,campaign_name,campaign_phase,user_past_sentiment_avg,user_engagement_growth,buzz_change_rate
0,kcqbs6hxybia,2024-12-09 11:26:15,Monday,Instagram,user_52nwb0a6,"Melbourne, Australia",pt,Just tried the Chromebook from Google. Best pu...,#Food,NaN,...,701,18991,0.19319,Google,Chromebook,BlackFriday,Launch,0.0953,-0.3672,19.1
1,vkmervg4ioos,2024-07-28 19:59:26,Sunday,Twitter,user_ucryct98,"Tokyo, Japan",ru,Just saw an ad for Microsoft Surface Laptop du...,"#MustHave, #Food","@CustomerService, @BrandCEO",...,359,52764,0.05086,Microsoft,Surface Laptop,PowerRelease,Post-Launch,0.1369,-0.4510,-42.6
2,memhx4o1x6yu,2024-11-23 14:00:12,Saturday,Reddit,user_7rrev126,"Beijing, China",ru,What's your opinion about Nike's Epic React? ...,"#Promo, #Food, #Trending",NaN,...,643,8887,0.45425,Nike,Epic React,BlackFriday,Post-Launch,0.2855,-0.4112,17.4
3,bhyo6piijqt9,2024-09-16 04:35:25,Monday,YouTube,user_4mxuq0ax,"Lagos, Nigeria",en,Bummed out with my new Diet Pepsi from Pepsi! ...,"#Reviews, #Sustainable","@StyleGuide, @BrandSupport",...,743,6696,0.42293,Pepsi,Diet Pepsi,LaunchWave,Launch,-0.2094,-0.0167,-5.5
4,c9dkiomowakt,2024-09-05 21:03:01,Thursday,Twitter,user_l1vpox2k,"Berlin, Germany",hi,Just tried the Corolla from Toyota. Absolutely...,"#Health, #Travel","@BrandSupport, @InfluencerName",...,703,47315,0.08773,Toyota,Corolla,LocalTouchpoints,Launch,0.6867,0.0807,38.8


In [10]:
data.shape # Dataset Shape

(12000, 28)

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   post_id                  12000 non-null  object 
 1   timestamp                12000 non-null  object 
 2   day_of_week              12000 non-null  object 
 3   platform                 12000 non-null  object 
 4   user_id                  12000 non-null  object 
 5   location                 12000 non-null  object 
 6   language                 12000 non-null  object 
 7   text_content             12000 non-null  object 
 8   hashtags                 12000 non-null  object 
 9   mentions                 8059 non-null   object 
 10  keywords                 12000 non-null  object 
 11  topic_category           12000 non-null  object 
 12  sentiment_score          12000 non-null  float64
 13  sentiment_label          12000 non-null  object 
 14  emotion_type          

In [12]:
data.describe()

,sentiment_score,toxicity_score,likes_count,shares_count,comments_count,impressions,engagement_rate,user_past_sentiment_avg,user_engagement_growth,buzz_change_rate
count,12000.000000,12000.000000,12000.00000,12000.000000,12000.00000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000
mean,0.000553,0.503868,2490.72025,1007.167167,504.34575,49811.338500,0.278137,0.001472,0.000998,0.729692
std,0.583563,0.288198,1441.53253,575.072282,288.68416,28930.289451,1.149206,0.576627,0.289940,57.787219
min,-0.999800,0.000000,0.00000,0.000000,0.00000,130.000000,0.001880,-0.999600,-0.499900,-99.900000
25%,-0.503200,0.251400,1236.00000,510.000000,253.00000,24716.500000,0.049100,-0.495975,-0.248400,-48.700000
50%,-0.006200,0.505950,2496.00000,1018.000000,503.00000,49674.000000,0.080605,0.001950,0.002800,0.900000
75%,0.513525,0.756200,3723.25000,1501.000000,755.00000,74815.000000,0.163123,0.501725,0.250700,50.100000
max,0.999900,0.999900,5000.00000,2000.000000,1000.00000,99997.000000,32.211710,0.999400,0.499900,99.900000


In [13]:
data.dtypes

post_id                     object
timestamp                   object
day_of_week                 object
platform                    object
user_id                     object
location                    object
language                    object
text_content                object
hashtags                    object
mentions                    object
keywords                    object
topic_category              object
sentiment_score            float64
sentiment_label             object
emotion_type                object
toxicity_score             float64
likes_count                  int64
shares_count                 int64
comments_count               int64
impressions                  int64
engagement_rate            float64
brand_name                  object
product_name                object
campaign_name               object
campaign_phase              object
user_past_sentiment_avg    float64
user_engagement_growth     float64
buzz_change_rate           float64
dtype: object

# **Structured Data Preprocessing**

## Check Missing and Duplicate Values

In [14]:
print(data.isnull().sum()) # Count missing values per column
print(data.isna().sum())

post_id                       0
timestamp                     0
day_of_week                   0
platform                      0
user_id                       0
location                      0
language                      0
text_content                  0
hashtags                      0
mentions                   3941
keywords                      0
topic_category                0
sentiment_score               0
sentiment_label               0
emotion_type                  0
toxicity_score                0
likes_count                   0
shares_count                  0
comments_count                0
impressions                   0
engagement_rate               0
brand_name                    0
product_name                  0
campaign_name                 0
campaign_phase                0
user_past_sentiment_avg       0
user_engagement_growth        0
buzz_change_rate              0
dtype: int64
post_id                       0
timestamp                     0
day_of_week                

In [15]:
# Drop missing values
data = data.dropna()
print(data.isnull().sum())

post_id                    0
timestamp                  0
day_of_week                0
platform                   0
user_id                    0
location                   0
language                   0
text_content               0
hashtags                   0
mentions                   0
keywords                   0
topic_category             0
sentiment_score            0
sentiment_label            0
emotion_type               0
toxicity_score             0
likes_count                0
shares_count               0
comments_count             0
impressions                0
engagement_rate            0
brand_name                 0
product_name               0
campaign_name              0
campaign_phase             0
user_past_sentiment_avg    0
user_engagement_growth     0
buzz_change_rate           0
dtype: int64


In [16]:
print("Data Rows Duplicates: ", data.duplicated().sum())
print("Post ID Duplicates: ", data["post_id"].duplicated().sum())

Data Rows Duplicates:  0
Post ID Duplicates:  0


## Remove Irrelevant Columns

In [17]:
data = data.drop(['post_id', 'user_id', 'hashtags', 'mentions', 'keywords'], axis=1)
print("New Shape with reduced columns: ", data.shape)

New Shape with reduced columns:  (8059, 23)


## Outlier Detection & Deletion

### IQR - Zscore Method Implementations

In [18]:
def remove_outliers(dataset = data, method='iqr', threshold=3):
  data_clean = data.copy()
  numeric_cols = data.select_dtypes(include='number').columns #Select Numerical columns

  if method == 'iqr':
    Q1 = data_clean[numeric_cols].quantile(0.25)
    Q3 = data_clean[numeric_cols].quantile(0.75)
    IQR = Q3 - Q1
    mask = ~((data_clean[numeric_cols] < (Q1 - 1.5 * IQR)) | (data_clean[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)
  elif method == 'zscore':
    z = np.abs(stats.zscore(data_clean[numeric_cols]))
    mask = (z < threshold).all(axis =1)
  data_filtered = data_clean[mask]
  print(f"Removed {len(data_clean) - len(data_filtered)} outliers using the {method.upper()} method")
  return data_filtered

In [19]:
data_clean = remove_outliers(data, method='iqr', threshold = 3)

Removed 997 outliers using the IQR method


## Dropping Sentiment Analysis Predictor Columns
- BERT handles encoding using its own tokenizer internally


In [20]:
# Take the clean data after removing outliers, nulls, etc
unstructured_data = data_clean[['text_content', 'sentiment_score', 'sentiment_label', 'emotion_type']]
data_clean = data_clean.drop(['text_content', 'sentiment_score', 'sentiment_label', 'emotion_type'], axis=1)

## Categorical Variable Encoding

In [21]:
# List all categorical-based columns
categorical_cols = data_clean.select_dtypes(include='object').columns
print("Categorical Columns: ", categorical_cols)

Categorical Columns:  Index(['timestamp', 'day_of_week', 'platform', 'location', 'language',
       'topic_category', 'brand_name', 'product_name', 'campaign_name',
       'campaign_phase'],
      dtype='object')


In [22]:
""" Encoding:
    - day_of_week,
    - platform,
    - location,
    - language,
    - text_content,
    - topic category,
    - sentiment_label,
    - emotion_type,
    - brand name,
    - product name,
    - campaign name,
    - campaign phase
    - sentiment_label,

 Note: Columns - text_content, sentiment_label, emotion_type will be
  used in the sentiment analysis predictor
"""
def encode_columns(data=data, label_cols=None, ordinal_cols=None, ordinal_mapping=None):
  data_encoded = data_clean.copy()

  # Label encoding for nominal data
  if label_cols:
    le = LabelEncoder()
    for col in label_cols:
      data_encoded[col] = le.fit_transform(data_encoded[col].astype(str))
  # Ordinal encoding for ordinal data
  if ordinal_cols:
    mappings = []
    for col in ordinal_cols:
      if ordinal_mapping and col in ordinal_mapping:
        mappings.append
        (
          {
            'col': col,
            'mapping': [(v, i) for i, v in enumerate(ordinal_mapping[col])]
          }
        )
      else:
        print("Missing order mapping for ordinal encoding")
    oe = OrdinalEncoder(categories=[ordinal_mapping[col] for col in ordinal_cols])
    data_encoded[ordinal_cols] = oe.fit_transform(data_encoded[ordinal_cols])
  return data_encoded


In [23]:


"""
Example usage:
data = {
    'color': ['red', 'blue', 'green', 'blue', 'red'],
    'size': ['small', 'medium', 'large', 'medium', 'large'],
    'brand': ['A', 'B', 'A', 'C', 'B']
}

df = pd.DataFrame(data)

# Label encode "brand"
# Ordinal encode "size" with order small < medium < large
encoded_df = encode_columns(
    df,
    label_cols=['brand'],
    ordinal_cols=['size'],
    ordinal_mapping={'size': ['small', 'medium', 'large']}
)

print(encoded_df)

OUTPUT:
   color  size  brand
0    red   0.0      0
1   blue   1.0      1
2  green   2.0      0
3   blue   1.0      2
4    red   2.0      1
"""

'\nExample usage:\ndata = {\n    \'color\': [\'red\', \'blue\', \'green\', \'blue\', \'red\'],\n    \'size\': [\'small\', \'medium\', \'large\', \'medium\', \'large\'],\n    \'brand\': [\'A\', \'B\', \'A\', \'C\', \'B\']\n}\n\ndf = pd.DataFrame(data)\n\n# Label encode "brand"\n# Ordinal encode "size" with order small < medium < large\nencoded_df = encode_columns(\n    df,\n    label_cols=[\'brand\'],\n    ordinal_cols=[\'size\'],\n    ordinal_mapping={\'size\': [\'small\', \'medium\', \'large\']}\n)\n\nprint(encoded_df)\n\nOUTPUT:\n   color  size  brand\n0    red   0.0      0\n1   blue   1.0      1\n2  green   2.0      0\n3   blue   1.0      2\n4    red   2.0      1\n'

## Data Normalization: Min-Max Scaling
 - Applicable in neural networks
 - Distance-based models

In [24]:
""" Normalizing:
    - likes_count,
    - comments_count,
    - shares_count,
    - impressions,
    - buzz_change_rate
"""
def normalize_numeric_cols (data=data):
  specified_columns_to_normalize = data[['likes_count', 'comments_count', 'shares_count', 'buzz_change_rate']]
  scaler = MinMaxScaler()
  data_scaled = data.copy()
  data_scaled[specified_columns_to_normalize] = scaler.fit_transform(data[specified_columns_to_normalize])

  return data_scaled

# **Unstructured Data Preprocessing**

## Remove URLs

In [25]:
# Detect URLs in a DataFrame column
def has_urls(data=unstructured_data, column='text_content'):
  url_pattern = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
  mask = data[column].astype(str).str.contains(url_pattern, regex=True, na=False)
  # Returns if there exists any urls present in all data frame and in each row
  return mask.any(), mask

has_urls()

(np.False_,
 1        False
 4        False
 7        False
 8        False
 9        False
          ...  
 11994    False
 11996    False
 11997    False
 11998    False
 11999    False
 Name: text_content, Length: 7062, dtype: bool)

In [26]:
# Remove URLs in a DataFrame column (if necessary)
def remove_urls(data=unstructured_data, column='text_content'):
  url_pattern = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
  data_clean = data.copy()
  data_clean[column] = data_clean[column].astype(str).str.replace(url_pattern, '', regex=True)
  return data_clean

In [27]:
unstructured_data_clean = remove_urls()

## Remove @ mentions

In [28]:
# Detect @mentions in a DataFrame column
def has_mentions(data=unstructured_data_clean, column='text_content'):
  mention_pattern = re.compile(r'@\w+', re.IGNORECASE)
  mask = data[column].astype(str).str.contains(mention_pattern, regex=True, na=False)
  return mask.any(), mask

has_mentions()

(np.True_,
 1        False
 4        False
 7        False
 8        False
 9        False
          ...  
 11994    False
 11996    False
 11997    False
 11998    False
 11999    False
 Name: text_content, Length: 7062, dtype: bool)

In [29]:
# Remove @mentions in a DataFrame column (@mentions do exist)
def remove_urls(data=unstructured_data_clean, column='text_content'):
  url_pattern = re.compile(r'@\w+', re.IGNORECASE)
  data_clean = data.copy()
  data_clean[column] = data_clean[column].astype(str).str.replace(url_pattern, '', regex=True)
  return data_clean

unstructured_data_clean = remove_urls()

## Remove Hashtags but keep words

In [30]:
def has_hashtags(data=unstructured_data_clean, column='text_content'):
  hashtag_pattern = re.compile(r'#\w+', re.IGNORECASE)
  mask = data[column].astype(str).str.contains(hashtag_pattern, regex=True, na=False)
  return mask.any(), mask

In [31]:
def remove_hashtags(data=unstructured_data_clean, column='text_content', inplace= False):
  # Removes the '#' from hashtags in the specified column but keeps the words
  if not inplace:
    data = data.copy()
    data[column] = data[column].astype(str).apply(lambda x: re.sub(r'#(\w+)', r'\1', x))
    return data

unstructured_data_clean = remove_hashtags()

## Trim Whitespace

In [32]:
def trim_whitespace(data=unstructured_data_clean, column='text_content', inplace=False):
  # Rmoves leading, trailing and extra spaces within text
  # Ex: '   Hello  world   ' -> 'Hello world'
  if not inplace:
    data = data.copy()
  data[column] = (
      data[column]
      .astype(str)
      .apply(lambda x: re.sub(r'\s+', ' ', x.strip()))
  )
  return data

unstructured_data_clean = trim_whitespace()

# **Sentiment Analysis Predictor**

### Encode Labels

In [33]:
sentiment_encoder = LabelEncoder()
emotion_encoder = LabelEncoder()

def get_encoders():
  sentiment_encoder = LabelEncoder()
  emotion_encoder = LabelEncoder()
  return sentiment_encoder, emotion_encoder

def encode_labels(data=unstructured_data_clean):
  data_encoded = data.copy()
  data_encoded['sentiment_label_enc'] = sentiment_encoder.fit_transform(unstructured_data_clean['sentiment_label'])
  data_encoded['emotion_type_enc'] = emotion_encoder.fit_transform(unstructured_data_clean['emotion_type'])

  return data_encoded

unstructured_data_encoded = encode_labels()

### Tokenize feature

In [34]:
# Initiating a DistilBERT Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

c:\Users\ASUS\Desktop\LLM Training Demo\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [35]:
def tokenize_function(row):
  return tokenizer(
      row["text_content"],
      padding="max_length",
      truncation=True,
      max_length=128
  )

# Creates a single column data frame (Tokenized text_content)
tokenized_column = unstructured_data_encoded.apply(tokenize_function, axis=1)
# Merge column with encoded data
tokenized_data = unstructured_data_encoded.copy()
tokenized_data['tokenized_text_content'] = tokenized_column

In [36]:
tokenized_data.head()

,text_content,sentiment_score,sentiment_label,emotion_type,sentiment_label_enc,emotion_type_enc,tokenized_text_content
1,Just saw an ad for Microsoft Surface Laptop du...,-0.3793,Negative,Angry,0,0,"[input_ids, attention_mask]"
4,Just tried the Corolla from Toyota. Absolutely...,0.5460,Positive,Happy,2,3,"[input_ids, attention_mask]"
7,Just saw an ad for Coca-Cola Coke Zero during ...,0.2034,Positive,Sad,2,4,"[input_ids, attention_mask]"
8,Just tried the Coke Zero from Coca-Cola. Absol...,0.8189,Positive,Excited,2,2,"[input_ids, attention_mask]"
9,My one week review of Google Pixel Watch: Disa...,-0.6546,Negative,Excited,0,2,"[input_ids, attention_mask]"


## Building Multi-Task Model for DistilBERT

In [37]:
"""
import torch
import torch.nn as nn
from transformers import BertModel

class MultiTaskBERT(nn.Module):
    def __init__(self, model_name, num_sentiment_classes, num_emotion_classes):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Regression head
        self.regressor = nn.Linear(hidden_size, 1)
        # Sentiment classification head
        self.sentiment_classifier = nn.Linear(hidden_size, num_sentiment_classes)
        # Emotion classification head
        self.emotion_classifier = nn.Linear(hidden_size, num_emotion_classes)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [batch, hidden_size]

        score = self.regressor(pooled).squeeze(-1)
        sentiment_logits = self.sentiment_classifier(pooled)
        emotion_logits = self.emotion_classifier(pooled)

        return score, sentiment_logits, emotion_logits

"""

'\nimport torch\nimport torch.nn as nn\nfrom transformers import BertModel\n\nclass MultiTaskBERT(nn.Module):\n    def __init__(self, model_name, num_sentiment_classes, num_emotion_classes):\n        super().__init__()\n        self.bert = BertModel.from_pretrained(model_name)\n        hidden_size = self.bert.config.hidden_size\n\n        # Regression head\n        self.regressor = nn.Linear(hidden_size, 1)\n        # Sentiment classification head\n        self.sentiment_classifier = nn.Linear(hidden_size, num_sentiment_classes)\n        # Emotion classification head\n        self.emotion_classifier = nn.Linear(hidden_size, num_emotion_classes)\n\n    def forward(self, input_ids, attention_mask=None):\n        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)\n        pooled = outputs.pooler_output  # [batch, hidden_size]\n\n        score = self.regressor(pooled).squeeze(-1)\n        sentiment_logits = self.sentiment_classifier(pooled)\n        emotion_logits = se

In [38]:
class MultiTaskDistilBERT(nn.Module):
  def __init__(self, model_name, num_sentiment_classes, num_emotion_classes):
    super().__init__()
    self.bert = DistilBertModel.from_pretrained(model_name)
    hidden_size = self.bert.config.hidden_size

    # Reggression head for sentiment_score
    self.regressor = nn.Linear(hidden_size, 1)
    # Classification head for sentiment_value
    self.sentiment_classifier = nn.Linear(hidden_size, num_sentiment_classes)
    # Classification head for emotion_type
    self.emotion_classifier = nn.Linear(hidden_size, num_emotion_classes)

  def forward(self, input_ids, attention_mask):
    outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
    hidden_state = outputs.last_hidden_state  #(batch, seq_len, hidden_size)
    pooled = hidden_state[:, 0] # Take [CLS] token

    # cls_embedding = hidden_state[:, 0]

    score = self.regressor(pooled).squeeze(-1)
    sentiment_logits = self.sentiment_classifier(pooled)
    emotion_logits = self.emotion_classifier(pooled)

    return score, sentiment_logits, emotion_logits


In [39]:
torch.device("cuda")

device(type='cuda')

### Inject Data

In [40]:
input_ids = torch.tensor(tokenized_data['tokenized_text_content'].apply(lambda x: x['input_ids']).tolist())
attention_masks = torch.tensor(tokenized_data['tokenized_text_content'].apply(lambda x: x['attention_mask']).tolist())
scores = torch.tensor(tokenized_data['sentiment_score'].values, dtype=torch.float32)
sentiment_labels = torch.tensor(tokenized_data['sentiment_label_enc'].values)
emotion_labels = torch.tensor(tokenized_data['emotion_type_enc'].values)

dataset = TensorDataset(input_ids, attention_masks, scores, sentiment_labels, emotion_labels)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = MultiTaskDistilBERT("bert-base-uncased",
                      num_sentiment_classes=len(sentiment_encoder.classes_),
                      num_emotion_classes=len(emotion_encoder.classes_)
                      )
# optimizer = optim.AdamW(model.parameters(), lr=2e-5)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01,      # prevents overfitting with many classes
    betas=(0.9, 0.999),
    eps=1e-8
)


c:\Users\ASUS\Desktop\LLM Training Demo\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You are using a model of type bert to instantiate a model of type distilbert. This is not supported for all

In [45]:
# ===== LOAD CHECKPOINT =====
resume = True  # set to False when starting fresh

if resume:
    checkpoint = torch.load("checkpoint/bert_checkpoint.pt", map_location="cuda") # 'cuda' if got gpu, else use 'cpu'
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
else:
    start_epoch = 0

# Create an empty list to store metrics per epoch
metrics_list = []

epoch_size = 1
for epoch in range(start_epoch, start_epoch + epoch_size):
    model.train()

    # Metric storage
    all_sent_pred = []
    all_sent_true = []

    all_emo_pred = []
    all_emo_true = []

    all_score_pred = []
    all_score_true = []

    loop = tqdm(loader, leave=True)
    loop.set_description(f"Epoch {epoch+1}/{start_epoch + epoch_size}")

    for batch in loop:
        input_ids, attention_mask, score, sent_label, emo_label = [b for b in batch]

        pred_score, sent_logits, emo_logits = model(input_ids, attention_mask)

        # Losses
        loss_reg = F.mse_loss(pred_score, score)
        loss_sent = F.cross_entropy(sent_logits, sent_label)
        loss_emo = F.cross_entropy(emo_logits, emo_label)

        loss = loss_reg + loss_sent + loss_emo

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # ---- Predictions for metrics ----
        # Classification
        all_sent_pred.extend(
            torch.argmax(sent_logits, dim=1).detach().cpu().numpy().tolist()
        )
        all_sent_true.extend(
            sent_label.detach().cpu().numpy().tolist()
        )

        all_emo_pred.extend(
            torch.argmax(emo_logits, dim=1).detach().cpu().numpy().tolist()
        )
        all_emo_true.extend(
            emo_label.detach().cpu().numpy().tolist()
        )

        # Regression
        all_score_pred.extend(
            pred_score.squeeze().detach().cpu().numpy().tolist()
        )
        all_score_true.extend(
            score.squeeze().detach().cpu().numpy().tolist()
        )

    # ========= METRICS PER EPOCH =========

    # Regression: RMSE, MAE, R²
    rmse = mean_squared_error(all_score_true, all_score_pred) ** 0.5
    mae = mean_absolute_error(all_score_true, all_score_pred)
    r2 = r2_score(all_score_true, all_score_pred)

    # Classification: Sentiment Label
    sent_acc = accuracy_score(all_sent_true, all_sent_pred)
    sent_prec, sent_rec, sent_f1, _ = precision_recall_fscore_support(
        all_sent_true, all_sent_pred, average="weighted", zero_division=0
    )

    # Classification: Emotion Type
    emo_acc = accuracy_score(all_emo_true, all_emo_pred)
    emo_prec, emo_rec, emo_f1, _ = precision_recall_fscore_support(
        all_emo_true, all_emo_pred, average="weighted", zero_division=0
    )

    # ========= PRINT METRICS =========
    print(f"\n================ Epoch {epoch+1} Metrics ================")
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")
    print(f"Sentiment - Acc: {sent_acc:.4f}, Prec: {sent_prec:.4f}, Rec: {sent_rec:.4f}, F1: {sent_f1:.4f}")
    print(f"Emotion   - Acc: {emo_acc:.4f}, Prec: {emo_prec:.4f}, Rec: {emo_rec:.4f}, F1: {emo_f1:.4f}")
    print("==========================================================\n")

    # ========= STORE METRICS =========
    metrics_list.append({
        "Epoch": epoch+1,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Sent_Acc": sent_acc,
        "Sent_Prec": sent_prec,
        "Sent_Rec": sent_rec,
        "Sent_F1": sent_f1,
        "Emo_Acc": emo_acc,
        "Emo_Prec": emo_prec,
        "Emo_Rec": emo_rec,
        "Emo_F1": emo_f1
    })

metrics_data = pd.DataFrame(metrics_list)
metrics_data.to_csv("BERT_training_metric.csv", index=False)
print("Metrics saved to BERT_training_metric.csv")

Epoch 5/5: 100%|██████████| 883/883 [18:53<00:00,  1.28s/it]


================ Epoch 5 Metrics ================
RMSE: 0.3055, MAE: 0.2380, R²: 0.7254
Sentiment - Acc: 0.9444, Prec: 0.9447, Rec: 0.9444, F1: 0.9443
Emotion   - Acc: 0.3840, Prec: 0.3849, Rec: 0.3840, F1: 0.3838

Metrics saved to BERT_training_metric.csv


### Saving Model Checkpoint

CUDA Usage

In [ ]:
# model.to("cuda")

In [46]:
save_path = "checkpoint/"
os.makedirs(save_path, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": epoch
}, save_path + "bert_checkpoint.pt")

### Predict Sample

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
text = "i am not happy"
tokens = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128).to(device)

model.eval()
with torch.no_grad():
    score, sent_logits, emo_logits = model(**tokens)

sentiment = sentiment_encoder.inverse_transform([sent_logits.argmax(dim=1).item()])[0]
emotion = emotion_encoder.inverse_transform([emo_logits.argmax(dim=1).item()])[0]

print(f"Score: {score.item():.3f}, Sentiment: {sentiment}, Emotion: {emotion}")


Score: 0.132, Sentiment: Positive, Emotion: Happy


# Export Model

In [ ]:
torch.save(model.state_dict(), "distilbert_multitask.pt")
joblib.dump(sentiment_encoder, "sentiment_encoder.pkl")
joblib.dump(emotion_encoder, "emotion_encoder.pkl")


## Load Model

In [ ]:
model.load_state_dict(torch.load("distilbert_multitask.pt"))
sentiment_encoder = joblib.load("sentiment_encoder.pkl")
emotion_encoder = joblib.load("emotion_encoder.pkl")
print("Finished Loading")

In [ ]:
# Evaluate Model
model.eval()

### Load Tokenizer

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

### Creating Inference function
This takes raw text → tokenizer → model → decode predictions.

In [ ]:
"""
import torch
import torch.nn.functional as F

def predict(text):
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.no_grad():
        score, sentiment_logits, emotion_logits = model(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"]
        )

    # Convert predictions
    predicted_score = score.item()

    predicted_sentiment = sentiment_encoder.inverse_transform(
        [torch.argmax(sentiment_logits).item()]
    )[0]

    predicted_emotion = emotion_encoder.inverse_transform(
        [torch.argmax(emotion_logits).item()]
    )[0]

    return {
        "sentiment_score": predicted_score,
        "sentiment_value": predicted_sentiment,
        "emotion_type": predicted_emotion
    }
"""


In [ ]:
#Example use
# print(predict("I absolutely love this product!"))